# 🧠 Multi-Tool Medical AI Agent
## Building an AI Agent to Interact with Medical Datasets and Web

This notebook creates:
- SQLite databases from medical datasets
- Database-specific query tools
- Web search tool for general medical knowledge
- Main AI agent that routes queries intelligently

## 📦 Step 1: Install Required Libraries

In [41]:
!pip install --upgrade --force-reinstall langchain langchain-community langchain-openai langgraph

^C


  Using cached langchain-1.2.13-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_openai-1.1.11-py3-none-any.whl.metadata (3.1 kB)
  Using cached langgraph-1.1.3-py3-none-any.whl.metadata (7.4 kB)
  Using cached langchain_core-1.2.20-py3-none-any.whl.metadata (4.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached langgraph_checkpoint-4.0.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.12-py3-none-any.whl.metadata (1.6 kB)
  Using cached xxhash-3.6.0-cp312-cp312-win_amd64.whl.metadata (13 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langsmith-0.7.22-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.0.2 requires pydantic<=2.12.4,>=2.11.10, but you have pydantic 2.12.5 which is incompatible.
kubernetes 34.1.0 requires urllib3<2.4.0,>=1.24.2, but yo

In [43]:
pip install -U langgraph langchain-openai langchain-community

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install -U langchain langchain-community langchain-openai langgraph


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install openai langchain langchain-openai langchain-community pandas sqlite3 kaggle python-dotenv tavily-python gradio kaggle

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement sqlite3 (from versions: none)
ERROR: No matching distribution found for sqlite3


In [7]:
pip install kaggle


   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   ------------------------------ --------- 3/4 [kaggle]
   ---------------------------------------- 4/4 [kaggle]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔑 Step 2: Setup API Keys

Create a `.env` file in the same directory with:
```
OPENAI_API_KEY=your_openai_api_key_here
TAVILY_API_KEY=your_tavily_api_key_here
KAGGLE_USERNAME=your_kaggle_username
KAGGLE_KEY=your_kaggle_key
```

In [56]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API keys are loaded
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

print("✅ API Keys loaded successfully!" if OPENAI_API_KEY else " Please add your API keys to .env file")

✅ API Keys loaded successfully!


## 🗄️ Step 4: Convert CSVs to SQLite Databases

In [57]:
import pandas as pd
import sqlite3
import glob

def create_database(csv_path, db_name, table_name):
    """
    Convert CSV to SQLite database
    """
    try:
        # Read CSV
        df = pd.read_csv(csv_path)
        
        # Create SQLite connection
        conn = sqlite3.connect(f'data/{db_name}')
        
        # Write to database
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        
        # Verify
        cursor = conn.cursor()
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        
        print(f" Created {db_name} with {count} rows in table '{table_name}'")
        print(f"   Columns: {', '.join(df.columns.tolist())}")
        
        conn.close()
        return True
    except Exception as e:
        print(f" Error creating {db_name}: {str(e)}")
        return False

# Find CSV files in each directory
heart_csv = glob.glob('data/heart/*.csv')[0]
cancer_csv = glob.glob('data/cancer/*.csv')[0]
diabetes_csv = glob.glob('data/diabetes/*.csv')[0]

# Create databases
create_database(heart_csv, 'heart_disease.db', 'heart_disease')
create_database(cancer_csv, 'cancer.db', 'cancer_data')
create_database(diabetes_csv, 'diabetes.db', 'diabetes_data')

print("\n All databases created successfully!")

 Created heart_disease.db with 1025 rows in table 'heart_disease'
   Columns: age, sex, cp, trestbps, chol, fbs, restecg, thalach, exang, oldpeak, slope, ca, thal, target
 Created cancer.db with 1500 rows in table 'cancer_data'
   Columns: Age, Gender, BMI, Smoking, GeneticRisk, PhysicalActivity, AlcoholIntake, CancerHistory, Diagnosis
 Created diabetes.db with 768 rows in table 'diabetes_data'
   Columns: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, Outcome

 All databases created successfully!


## 🔍 Step 5: Explore Database Schemas

In [58]:
import sqlite3
import pandas as pd

def explore_database(db_path, table_name):
    """
    Display database schema and sample data
    """
    conn = sqlite3.connect(db_path)
    
    # Get schema
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name})")
    schema = cursor.fetchall()
    
    print(f"\n{'='*60}")
    print(f"Database: {db_path}")
    print(f"Table: {table_name}")
    print(f"{'='*60}")
    print("\nSchema:")
    for col in schema:
        print(f"  - {col[1]} ({col[2]})")
    
    # Get sample data
    df = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 3", conn)
    print("\nSample Data:")
    print(df)
    
    conn.close()

explore_database('data/heart_disease.db', 'heart_disease')
explore_database('data/cancer.db', 'cancer_data')
explore_database('data/diabetes.db', 'diabetes_data')


Database: data/heart_disease.db
Table: heart_disease

Schema:
  - age (INTEGER)
  - sex (INTEGER)
  - cp (INTEGER)
  - trestbps (INTEGER)
  - chol (INTEGER)
  - fbs (INTEGER)
  - restecg (INTEGER)
  - thalach (INTEGER)
  - exang (INTEGER)
  - oldpeak (REAL)
  - slope (INTEGER)
  - ca (INTEGER)
  - thal (INTEGER)
  - target (INTEGER)

Sample Data:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  

Database: data/cancer.db
Table: cancer_data

Schema:
  - Age (INTEGER)
  - Gender (INTEGER)
  - BMI (REAL)
  - Smoking (INTEGER)
  - GeneticRisk (INTEGER)
  - PhysicalActivity (REAL)
  - AlcoholIntake (REAL)
  - CancerHistory (INTEGER)


## 🛠️ Step 6: Build Database-Specific Tools

In [59]:
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
try:
    from langchain.agents import AgentType
except ImportError:
    # Jodi AgentType na thake, tobe chinta nei, amra direct string use korbo niche
    AgentType = None 

try:
    from langchain_core.tools import Tool
except ImportError:
    from langchain.tools import Tool



# Initialize OpenAI LLM
llm = ChatOpenAI(
    model="gpt-4.1-nano",  
    temperature=0,
    openai_api_key=OPENAI_API_KEY
)

class DatabaseQueryTool:
    """
    A tool that queries a specific medical database
    """
    def __init__(self, db_path, table_name, description):
        self.db_path = db_path
        self.table_name = table_name
        self.description = description
        
        # Create SQL database connection
        self.db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
        
        # Create SQL agent
        self.toolkit = SQLDatabaseToolkit(db=self.db, llm=llm)
        self.agent = create_sql_agent(
            llm=llm,
            toolkit=self.toolkit,
            agent_type="zero-shot-react-description",
            verbose=True
        )
    
    def query(self, question: str) -> str:
        """
        Execute a natural language query on the database
        """
        try:
            result = self.agent.invoke({"input": question})
            return result['output']
        except Exception as e:
            return f"Error querying database: {str(e)}"

# Create database tools
print(" Creating Heart Disease DB Tool...")
heart_disease_tool = DatabaseQueryTool(
    db_path="data/heart_disease.db",
    table_name="heart_disease",
    description="Query the heart disease dataset for statistics, patient data, and heart disease patterns"
)

print(" Creating Cancer DB Tool...")
cancer_tool = DatabaseQueryTool(
    db_path="data/cancer.db",
    table_name="cancer_data",
    description="Query the cancer prediction dataset for statistics, patient data, and cancer patterns"
)

print(" Creating Diabetes DB Tool...")
diabetes_tool = DatabaseQueryTool(
    db_path="data/diabetes.db",
    table_name="diabetes_data",
    description="Query the diabetes dataset for statistics, patient data, and diabetes patterns"
)

print("\n All database tools created successfully!")

 Creating Heart Disease DB Tool...
 Creating Cancer DB Tool...
 Creating Diabetes DB Tool...

 All database tools created successfully!


## 🌐 Step 7: Build Web Search Tool

In [61]:
from tavily import TavilyClient

class MedicalWebSearchTool:
    """
    A tool for searching medical information on the web
    """
    def __init__(self, api_key):
        self.client = TavilyClient(api_key=api_key)
        self.description = "Search the web for general medical knowledge, definitions, symptoms, treatments, and cures"
    
    def search(self, query: str) -> str:
        """
        Search the web for medical information
        """
        try:
            # Enhance query with medical context
            medical_query = f"medical information: {query}"
            
            # Perform search
            response = self.client.search(
                query=medical_query,
                search_depth="advanced",
                max_results=5
            )
            
            # Extract and format results
            results = []
            for result in response.get('results', []):
                results.append(f"**{result['title']}**\n{result['content']}\n")
            
            if results:
                return "\n".join(results)
            else:
                return "No relevant medical information found."
        except Exception as e:
            return f"Error performing web search: {str(e)}"

# Create web search tool
print(" Creating Medical Web Search Tool...")
web_search_tool = MedicalWebSearchTool(api_key=TAVILY_API_KEY)
print(" Web search tool created successfully!")

 Creating Medical Web Search Tool...
 Web search tool created successfully!


## 🤖 Step 8: Build Main AI Agent with Tool Routing

In [63]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class MedicalAIAgent:
    def __init__(self, heart_tool, cancer_tool, diabetes_tool, web_tool, llm):
        # Tools Reference
        self.tools = {
            "HeartDiseaseDBTool": heart_tool.query,
            "CancerDBTool": cancer_tool.query,
            "DiabetesDBTool": diabetes_tool.query,
            "MedicalWebSearchTool": web_tool.search
        }
        self.llm = llm
        
        # 1. Router Prompt: LLM sudhu tool-er nam bolbe
        self.router_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a routing assistant. Based on the user's question, pick the BEST tool name from this list:
            - HeartDiseaseDBTool (For heart statistics/data)
            - CancerDBTool (For cancer statistics/data)
            - DiabetesDBTool (For diabetes statistics/data)
            - MedicalWebSearchTool (For general medical knowledge, symptoms, cures)

            ONLY return the tool name. Nothing else."""),
            ("human", "{input}")
        ])
        
        self.router_chain = self.router_prompt | self.llm | StrOutputParser()

    def query(self, question: str) -> str:
        print(f"🤔 Thinking about: {question}")
        try:
            # Step 1: Decide which tool to use
            selected_tool_name = self.router_chain.invoke({"input": question}).strip()
            print(f"🛠️ Selected Tool: {selected_tool_name}")

            # Step 2: Call the selected tool
            if selected_tool_name in self.tools:
                tool_func = self.tools[selected_tool_name]
                result = tool_func(question)
                
                # Step 3: Format the final answer using LLM
                final_answer_prompt = ChatPromptTemplate.from_template(
                    "User asked: {question}\nTool result: {result}\n\nPlease provide a clear, medical summary based on this data."
                )
                final_chain = final_answer_prompt | self.llm | StrOutputParser()
                return final_chain.invoke({"question": question, "result": result})
            else:
                return "I'm sorry, I couldn't decide which medical database to use. Please be more specific."

        except Exception as e:
            return f"Error: {str(e)}"

# Create the main AI agent
print("🤖 Creating Custom Medical AI Agent (Router Mode)...")
medical_agent = MedicalAIAgent(
    heart_tool=heart_disease_tool,
    cancer_tool=cancer_tool,
    diabetes_tool=diabetes_tool,
    web_tool=web_search_tool,
    llm=llm
)
print("✅ Ready to answer medical queries!")

🤖 Creating Custom Medical AI Agent (Router Mode)...
✅ Ready to answer medical queries!


## 🧪 Step 9: Test the Agent with Sample Queries

In [64]:
# Test database queries
print("\n" + "="*80)
print("TEST 1: Database Query - Heart Disease Statistics")
print("="*80)
response = medical_agent.query("How many patients in the heart disease dataset have heart disease?")
print("\n📊 Response:", response)


TEST 1: Database Query - Heart Disease Statistics
🤔 Thinking about: How many patients in the heart disease dataset have heart disease?

📊 Response: Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************19AA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


In [27]:
print("\n" + "="*80)
print("TEST 2: Web Search - Medical Definition")
print("="*80)
response = medical_agent.query("What is diabetes and what are its main symptoms?")
print("\n🌐 Response:", response)


TEST 2: Web Search - Medical Definition
🤔 Thinking about: What is diabetes and what are its main symptoms?

🌐 Response: Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************19AA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


In [ ]:
print("\n" + "="*80)
print("TEST 3: Database Query - Cancer Statistics")
print("="*80)
response = medical_agent.query("What is the average age of patients in the cancer dataset?")
print("\n📊 Response:", response)

In [ ]:
print("\n" + "="*80)
print("TEST 4: Web Search - Treatment Information")
print("="*80)
response = medical_agent.query("What are the common treatments for heart disease?")
print("\n🌐 Response:", response)

## 🎨 Step 10: Create Gradio Interface

In [ ]:
import gradio as gr

def query_medical_agent(question, history):
    """
    Process user query through the medical agent
    """
    response = medical_agent.query(question)
    return response

# Create Gradio interface
demo = gr.ChatInterface(
    fn=query_medical_agent,
    title="🧠 Medical AI Agent",
    description="""Ask questions about medical data or general medical knowledge!
    
    **Examples:**
    - 📊 Data Queries: "How many patients have diabetes?", "What's the average age in the cancer dataset?"
    - 🌐 General Knowledge: "What is hypertension?", "What are the symptoms of diabetes?"
    """,
    examples=[
        "How many patients in the heart disease dataset have heart disease?",
        "What is diabetes and what are its symptoms?",
        "What is the average age of cancer patients in the dataset?",
        "What are the risk factors for heart disease?",
        "Show me statistics about glucose levels in diabetic patients",
        "How is cancer typically treated?"
    ],
    theme=gr.themes.Soft(),
    retry_btn="🔄 Retry",
    undo_btn="↩️ Undo",
    clear_btn="🗑️ Clear"
)

# Launch the interface
if __name__ == "__main__":
    demo.launch(share=True, debug=True)

## 📝 Additional Utility Functions

In [ ]:
def get_database_stats():
    """
    Get statistics about all databases
    """
    stats = {}
    
    # Heart Disease
    conn = sqlite3.connect('data/heart_disease.db')
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM heart_disease")
    stats['heart_disease'] = cursor.fetchone()[0]
    conn.close()
    
    # Cancer
    conn = sqlite3.connect('data/cancer.db')
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM cancer_data")
    stats['cancer'] = cursor.fetchone()[0]
    conn.close()
    
    # Diabetes
    conn = sqlite3.connect('data/diabetes.db')
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM diabetes_data")
    stats['diabetes'] = cursor.fetchone()[0]
    conn.close()
    
    return stats

# Display statistics
stats = get_database_stats()
print("\n📊 Database Statistics:")
print(f"  - Heart Disease: {stats['heart_disease']} records")
print(f"  - Cancer: {stats['cancer']} records")
print(f"  - Diabetes: {stats['diabetes']} records")